# 8.288 Worksheet 1 — HE 1158-2313: radial velocity

## How this notebook fits together

In [ ]:
import json
import numpy as np

RES = "results/"
HDR   = json.load(open(RES + "part4-4_header.json"))
# §4.2.2 was run against both templates the worksheet offers
CCF_ALL = {t: json.load(open(f"{RES}part4-2-2_geocentric_rv_{t}.json"))
           for t in ("HD140283", "HD122563")}
ADOPTED_TEMPLATE = "HD122563"     # cool metal-poor giant: closest match to our star
CCF = CCF_ALL[ADOPTED_TEMPLATE]
SHIFT = json.load(open(RES + "part4-3_shift_and_helio.json"))

V_GEO_AF   = CCF["v_geo_kms"]                       # AF's route: HD 140283, H-alpha window
HELIO_CORR = SHIFT["helio_corr_adopted_kms"]        # at UT-START, from the header
GAIA, GAIA_E = 350.28, 0.19                          # DR3, for comparison only

for t, d in CCF_ALL.items():
    mark = "  <- adopted" if t == ADOPTED_TEMPLATE else ""
    print(f"§4.2.2 vs {t}: v_geo = {d['v_geo_kms']:+.2f} +/- {d['v_geo_err_kms']:.2f} km/s"
          f"  (mean of {len(d['per_window_kms']) - len(d['excluded'])} metal windows,"
          f" H-alpha excluded){mark}")
_vals = [d["v_geo_kms"] for d in CCF_ALL.values()]
_spread = max(_vals) - min(_vals)
print(f"        template-to-template spread = {_spread:.2f} km/s")
print(f"§4.4   correction at UT-START {HDR['UT-START']} = {HELIO_CORR:+.3f} km/s")
print(f"       header velocity keywords: {HDR['_velocity_keywords'] or 'none'}")
print(f"§4.3   rest-frame spectrum: {', '.join(SHIFT['rest_files'])}")

# used by the figures below
C = 299792.458
V_GEO = V_GEO_AF

In [ ]:
"""Worksheet 1 — HE 1158-2313 (MIKE, LCO, 2013-01-09): plots, RV by hand, RV by cross-correlation,
shift to rest, heliocentric correction. Run in the `8288` env from ~/courses/8.288/scripts."""
import numpy as np, matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
from astropy.io import fits
from IPython.display import display   # every saved figure is also shown below its cell
from astropy.time import Time
from astropy.coordinates import SkyCoord, EarthLocation
import astropy.units as u
from scipy.optimize import curve_fit
from scipy.ndimage import gaussian_filter1d
from PyAstronomy import pyasl

C = 299792.458
D = "../../../data/"; P = "plots/"
plt.rcParams.update({"font.size": 11, "axes.labelsize": 12, "figure.dpi": 110})

hdul = fits.open(D + "he1158-2313_uncor.fits"); hdr = hdul[1].header; t = hdul[1].data
w, f = np.asarray(t["wavelength"], float), np.asarray(t["norm_flux"], float)
o = np.argsort(w, kind="stable"); w, f = w[o], f[o]          # merged orders are not strictly monotonic
print(f"{hdr['OBJECT']}  RA={hdr['RA2000']} DEC={hdr['DEC2000']}  {hdr['DATE-OBS']}  {hdr['HIERARCH INSTRUMENT']}  {hdr['TELESCOP']}")
print(f"{len(w)} pixels, {w[0]:.0f}-{w[-1]:.0f} A, median step {np.median(np.diff(w)):.4f} A")

def cut(lo, hi, wave=w, flux=f):
    m = (wave > lo) & (wave < hi); return wave[m], flux[m]

## Part 4.2.1 — RV by hand

In [ ]:
lines_hand = {"Mg I 5167.321": 5167.321, "Mg I 5172.698": 5172.698, "Mg I 5183.619": 5183.619, "H-alpha 6562.801": 6562.801}
def gauss(x, a, mu, s, c): return c - a * np.exp(-0.5 * ((x - mu) / s) ** 2)
v_guess = 350.0                                  # first-look eyeball shift of H-alpha, refined below
hand, fit_hand = {}, {}
for name, lam0 in lines_hand.items():
    lam_obs_guess = lam0 * (1 + v_guess / C); half = 2.5 if "H-alpha" in name else 1.0
    x, y = cut(lam_obs_guess - half, lam_obs_guess + half)
    p, cov = curve_fit(gauss, x, y, p0=[1 - y.min(), x[np.argmin(y)], 0.3 if "Mg" in name else 0.8, 1.0])
    mu, emu = p[1], np.sqrt(cov[1, 1]); v = (mu - lam0) / lam0 * C
    hand[name] = (mu, v, emu / lam0 * C)
    fit_hand[name] = (p, cov, x, y, lam0)        # kept so the fits can be plotted below
    print(f"  {name:18s} core at {mu:9.3f} A  (dlam = {mu-lam0:+.3f})  ->  v_geo = {v:+8.2f} +/- {emu/lam0*C:.2f} km/s")
v_hand = np.array([hand[k][1] for k in hand if "Mg" in k])
print(f"  Mg b mean {v_hand.mean():+.2f} km/s, scatter (std) {v_hand.std(ddof=1):.2f} km/s; H-alpha {hand['H-alpha 6562.801'][1]:+.2f} km/s")
print("  redshifted (positive v): star moving AWAY from us")

In [ ]:
from matplotlib.ticker import MaxNLocator

# --- figure 1: the fits themselves
fig, axes = plt.subplots(2, 2, figsize=(11, 6.6))
for ax, (name, (pfit, cov, x, y, lam0)) in zip(axes.ravel(), fit_hand.items()):
    xm = np.linspace(x.min(), x.max(), 600)
    ax.plot(x, y, "o", ms=2.6, color="0.25", label="MIKE data")
    ax.plot(xm, gauss(xm, *pfit), "-", color="C0", lw=1.4, label="Gaussian fit")
    mu, emu = pfit[1], np.sqrt(cov[1, 1])
    rms = np.std(y - gauss(x, *pfit))
    ax.axvline(mu, color="C3", ls="--", lw=1.0)
    v = (mu - lam0) / lam0 * C
    # the rest wavelength is ~6 A blueward, off this panel; quote the offset instead of
    # stretching the axis to reach it (the zoom figures in Part 3 show it visually)
    ax.set_title(f"{name}\n$\\Delta\\lambda$ = {mu - lam0:+.3f} $\\AA$,  "
                 f"v = {v:+.2f} $\\pm$ {emu/lam0*C:.2f} km/s,  fit RMS {rms:.3f}",
                 loc="left", fontsize=9)
    if "5167" in name:
        lam_fe = 5167.49 * (1 + v / C)
        ax.axvline(lam_fe, color="C1", ls="-.", lw=1.1)
        ax.text(lam_fe + 0.06, 0.60, "Fe I 5167.49\nblend: centroid\nbiased redward",
                fontsize=7.5, color="C1", va="center", ha="left")
    ax.set_xlim(x.min(), x.max())
    ax.set_ylim(0, 1.25)
    ax.set_xlabel(r"$\lambda$ ($\AA$), observed")
    ax.set_ylabel("Normalised flux")
    ax.minorticks_on()
    ax.tick_params(direction="in", top=True, right=True, which="both")
    ax.xaxis.set_major_locator(MaxNLocator(nbins=5, steps=[1, 2, 5, 10]))
    ax.legend(loc="lower left", fontsize=7.5, frameon=False)
fig.suptitle("§4.2.1 by hand: Gaussian fits to the line cores (observed frame). "
             "Dashed red = fitted core; rest wavelength is ~6 $\\AA$ blueward, off panel.",
             fontsize=9.5, x=0.01, ha="left")
fig.tight_layout(rect=[0, 0, 1, 0.97])
fig.savefig(P + "byhand_line_fits.png", dpi=130)
display(fig); plt.close(fig)

# --- second figure: do the lines agree with each other, and with the §4.2.2 value?
fig, ax = plt.subplots(figsize=(8.6, 3.6))
names = list(fit_hand)
ys = np.arange(len(names))[::-1]
for yi, name in zip(ys, names):
    mu, v, err = hand[name]
    use = "5167" not in name
    ax.errorbar(v, yi, xerr=err, fmt="o", ms=6, capsize=3,
                color="k" if use else "C1", mfc="k" if use else "none", lw=1.2)
    if not use:
        ax.annotate("excluded (Fe I blend)", xy=(v, yi), xytext=(0, 10),
                    textcoords="offset points", ha="center", va="bottom",
                    fontsize=8, color="C1")
grid = CCF["rv_scan_kms"][2]
ax.axvspan(V_GEO_AF - grid, V_GEO_AF + grid, color="C0", alpha=0.20, zorder=0)
ax.axvline(V_GEO_AF, color="C0", lw=1.2, zorder=1,
           label=f"§4.2.2 cross-correlation vs HD 140283: {V_GEO_AF:+.2f} km/s ({grid} km/s grid)")
clean = np.array([hand[n][1] for n in names if "5167" not in n and "Mg" in n])
ax.axvline(clean.mean(), color="C2", ls="--", lw=1.2,
           label=f"clean Mg b mean {clean.mean():+.2f} km/s")
ax.set_yticks(ys); ax.set_yticklabels(names, fontsize=9)
ax.set_xlabel("geocentric radial velocity (km/s)")
ax.set_title("The by-hand lines against the independent cross-correlation", loc="left", fontsize=10)
ax.minorticks_on(); ax.tick_params(direction="in", top=True, right=True, which="both")
ax.legend(loc="lower right", fontsize=8, frameon=False)
fig.tight_layout()
fig.savefig(P + "byhand_velocity_summary.png", dpi=130)
display(fig); plt.close(fig)
print(f"  clean Mg b mean {clean.mean():+.2f}  |  §4.2.2 {V_GEO_AF:+.2f}"
      f"  |  difference {clean.mean()-V_GEO_AF:+.2f} km/s")

### Seeing the by-hand velocity by eye, using the Sun

In [ ]:
D, P = "../../../data/", "plots/"

# our star as observed, and the rest-frame product written by part4-3
t_obs = fits.open(D + "he1158-2313_uncor.fits")[1].data
w_obs = np.asarray(t_obs["wavelength"], float); f_obs = np.asarray(t_obs["norm_flux"], float)
_o = np.argsort(w_obs, kind="stable"); w_obs, f_obs = w_obs[_o], f_obs[_o]
w_rest = w_obs / (1 + V_GEO / C)

# the Sun, at rest, wavelengths in air; smoothed from R~400,000 to MIKE's R~28,000 so that
# depths and widths are comparable by eye rather than differing by resolution alone
from scipy.ndimage import gaussian_filter1d
t_sun = fits.open(D + "solar_fts_kpno.fits")[1].data
w_sun = np.asarray(t_sun["wavelength"], float)
_step = np.median(np.diff(w_sun))
f_sun = gaussian_filter1d(np.asarray(t_sun["norm_flux"], float),
                          (np.median(w_sun) / 28000.0) / 2.355 / _step)
print(f"star {len(w_obs)} px | Sun {len(w_sun)} px, smoothed to R~28000")
print(f"expected shift at 5183.62 A: {5183.62 * V_GEO / C:+.2f} A")

# G64-12, the §4.2.4 radial-velocity standard, as a second template. Both it and our star are
# MIKE spectra (R ~ 22000 and ~17000 measured on Mg I 5183.6), so no smoothing is needed - unlike
# the FTS solar atlas. Checked: this file is already at rest (H-alpha core at 6562.81 A).
from PyAstronomy import pyasl
w_g64, f_g64 = (np.asarray(x, float) for x in pyasl.read1dFitsSpec(D + "G64-12.fits"))
print(f"G64-12 {len(w_g64)} px, {w_g64[0]:.0f}-{w_g64[-1]:.0f} A")

# the two §4.2.2 reference stars, for the all-systems comparison further down
w_hd140, f_hd140 = (np.asarray(x, float) for x in pyasl.read1dFitsSpec(D + "HD140283.fits"))
w_hd122, f_hd122 = (np.asarray(x, float) for x in pyasl.read1dFitsSpec(D + "HD122563.fits"))
print(f"HD 140283 {len(w_hd140)} px | HD 122563 {len(w_hd122)} px")

In [ ]:
def zoom(name, lo, hi, marks, note="", tmpl=None, tname="HD 122563 (MIKE, at rest)",
         tcol="C4", prefix="zoom_"):
    """One region, two panels: as observed vs a template at rest, then shifted to rest.

    tmpl: (wavelength, flux) of the comparison spectrum. Defaults to HD 122563 - the same
    instrument as our spectrum (MIKE), and the template the adopted velocity comes from."""
    w_t, f_t = (w_hd122, f_hd122) if tmpl is None else tmpl
    fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
    for ax, (wx, lab, col) in zip(axes, [(w_obs, "HE 1158-2313 (as observed)", "C0"),
                                         (w_rest, f"HE 1158-2313 (shifted by {V_GEO:+.1f} km/s)", "k")]):
        ms = (w_t > lo - 3) & (w_t < hi + 3)
        ax.plot(w_t[ms], f_t[ms], color=tcol, lw=0.8, label=tname)
        m = (wx > lo - 3) & (wx < hi + 3)
        ax.plot(wx[m], f_obs[m], color=col, lw=1.0, label=lab)
        for lam, txt in marks.items():
            ax.axvline(lam, color="0.35", lw=0.7, ls="--")
            ax.text(lam, 1.10, txt, rotation=90, ha="center", va="bottom", fontsize=8, color="0.35")
        # on the observed panel, show the offset explicitly on the strongest marked line
        if wx is w_obs and marks:
            lam0 = list(marks)[0]
            lam1 = lam0 * (1 + V_GEO / C)
            ax.annotate("", xy=(lam1, 0.22), xytext=(lam0, 0.22),
                        arrowprops=dict(arrowstyle="->", color="C3", lw=1.1))
            ax.text((lam0 + lam1) / 2, 0.27,
                    f"{lam1 - lam0:+.2f} " + r"$\AA$" + f" = {V_GEO:+.0f} km/s",
                    ha="center", va="bottom", fontsize=8, color="C3")
        ax.set_xlim(lo, hi)
        ax.set_ylim(0, 1.28)
        ax.set_ylabel("Normalised flux")
        # course house style: ticks inward on all four sides, minor ticks on
        ax.minorticks_on()
        ax.tick_params(direction="in", top=True, right=True, which="both")
        ax.legend(loc="lower left", fontsize=8, frameon=False)
    axes[0].set_title(f"{name} — grey dashes: rest wavelengths, where {tname.split(' (')[0]}'s lines sit" +
                      (f"\n{note}" if note else ""), loc="left", fontsize=9.5)
    axes[1].set_xlabel(r"$\lambda$ ($\AA$)")
    fig.tight_layout()
    out = P + prefix + name.split()[0].lower().replace("$", "").replace("\\", "") + ".png"
    fig.savefig(out, dpi=130)
    display(fig)
    plt.close(fig)
    print("wrote", out)

REGIONS = [
 ("Mgb triplet", 5160, 5196, {5167.32: "Mg I", 5172.70: "Mg I", 5183.62: "Mg I"},
  "the classic region: three strong, sharp lines in both spectra"),
 ("NaD", 5884, 5900, {5889.95: "Na D$_2$", 5895.92: "Na D$_1$"}, ""),
 ("Halpha", 6548, 6578, {6562.80: r"H$\alpha$"}, "broad: good for a first eyeball, poor for precision"),
 ("Hbeta", 4850, 4874, {4861.35: r"H$\beta$"}, ""),
 ("CaIIK", 3924, 3944, {3933.66: "Ca II K"}, "saturated in both — depth carries no information here"),
 ("Gband", 4300, 4332, {4313.0: "CH", 4323.0: "CH"},
  "two cool metal-poor giants, same instrument: the CH lines correspond closely"),
]

for nm, lo, hi, marks, note in REGIONS:
    zoom(nm, lo, hi, marks, note)

### The same comparison against G64-12

In [ ]:
for nm, lo, hi, marks, note in REGIONS:
    zoom(nm, lo, hi, marks,
         note.replace("two cool metal-poor giants, same instrument: the CH lines correspond closely",
                      "warmer and more metal-poor than ours: far fewer lines, but the strong ones match"),
         tmpl=(w_g64, f_g64), tname="G64-12 (at rest)", tcol="C2", prefix="zoom_g64_")

## Part 4.2.2 — RV by cross-correlation

In [ ]:
print(f"§4.2.2 adopted result ({ADOPTED_TEMPLATE}), from {RES}part4-2-2_geocentric_rv_{ADOPTED_TEMPLATE}.json")
for k in ("v_geo_kms", "v_geo_err_kms", "window_scatter_kms", "per_window_kms", "excluded", "template", "rv_scan_kms"):
    print(f"   {k:18s} {CCF[k]}")
print(f"\n   grid step {CCF['rv_scan_kms'][2]} km/s sets the precision of this value")

In [ ]:
# §4.2.3 answered from the numbers, rather than asserted.
mg_clean = np.array([v for k, (mu, v, e) in hand.items() if "Mg" in k and "5167" not in k])
mg_err   = np.array([e for k, (mu, v, e) in hand.items() if "Mg" in k and "5167" not in k])
halpha   = hand["H-alpha 6562.801"]
rms = {k: float(np.std(y - gauss(x, *pf))) for k, (pf, cov, x, y, l0) in fit_hand.items()}

print("How different are the three measurements?")
print(f"  by hand, clean Mg b mean : {mg_clean.mean():+8.2f} km/s   (the two lines differ by {abs(np.diff(mg_clean)[0]):.2f})")
print(f"  by hand, H-alpha        : {halpha[1]:+8.2f} km/s")
print(f"  cross-corr, HD 122563   : {CCF_ALL['HD122563']['v_geo_kms']:+8.2f} km/s   <- adopted")
print(f"  cross-corr, HD 140283   : {CCF_ALL['HD140283']['v_geo_kms']:+8.2f} km/s")
print(f"  full range of the four  : {max(mg_clean.mean(), halpha[1], *[d['v_geo_kms'] for d in CCF_ALL.values()]) - min(mg_clean.mean(), halpha[1], *[d['v_geo_kms'] for d in CCF_ALL.values()]):.2f} km/s")
print()
print("Can we estimate a measurement uncertainty?  Three independent handles:")
print(f"  1. line-to-line, by hand      : {abs(np.diff(mg_clean)[0])/2:.2f} km/s (half the Mg b split)")
print(f"  2. template-to-template, CCF  : {abs(CCF_ALL['HD140283']['v_geo_kms'] - CCF_ALL['HD122563']['v_geo_kms']):.2f} km/s")
print(f"  3. zero point, from G64-12    : 0.60 km/s (§4.2.4 below)")
print(f"  ... but the methods differ by {abs(mg_clean.mean() - CCF_ALL['HD122563']['v_geo_kms']):.2f} km/s, which is LARGER than any of them.")
print("  So the honest uncertainty is set by the method difference, not by the internal scatter:")
print(f"     adopt +/- ~{abs(mg_clean.mean() - CCF_ALL['HD122563']['v_geo_kms'])/2:.1f} km/s, i.e. +/-0.5-1 km/s, not the +/-0.06 a single fit suggests.")
print()
print("Would the uncertainty be smaller or larger using H-alpha at ~6563 A?")
print(f"  formal fit error : H-alpha {halpha[2]:.2f} km/s vs Mg I 5183.6 {mg_err[-1]:.2f} km/s -> H-alpha is WORSE")
print(f"  fit residual RMS : H-alpha {rms['H-alpha 6562.801']:.3f} vs Mg I 5183.6 {rms['Mg I 5183.619']:.3f} -> a Gaussian fits H-alpha poorly")
print("  Two effects fight: dv = c*dlam/lam, so at 6563 A the same error in Angstroms buys a")
print("  SMALLER velocity error than at 5183 A - that favours H-alpha. But H-alpha is broad, and")
print("  in a giant its core forms in the chromosphere and is asymmetric, so the centroid is much")
print("  less well defined. The second effect wins here: the sharp Mg lines give the better value.")

In [ ]:
# Cross-check: the shift applied in §4.3 should move Mg I 5183.62 by 5183.62 * v/c = +6.04 A.
import numpy as np
obs  = fits.open(D + "he1158-2313_uncor.fits")[1].data
rest = fits.open(D + "he1158-2313_rest.fits")[1].data
w_o = np.sort(np.asarray(obs["wavelength"], float))
w_r = np.sort(np.asarray(rest["wavelength"], float))
i = np.argmin(np.abs(w_r - 5183.619))
print(f"observed lambda at that pixel : {w_o[i]:.3f} A")
print(f"rest-frame lambda             : {w_r[i]:.3f} A")
print(f"applied shift                 : {w_o[i]-w_r[i]:+.3f} A"
      f"  ({(w_o[i]-w_r[i])/5183.619*299792.458:+.2f} km/s)")

## Part 4.3 — the rest-frame spectrum

In [ ]:
# The rest-frame spectrum written by part4-3, reloaded from disk (not recomputed here).
t_rest = fits.open(D + "he1158-2313_rest.fits")[1].data
w_r = np.asarray(t_rest["wavelength"], float)
f_r = np.asarray(t_rest["norm_flux"], float)
_o = np.argsort(w_r, kind="stable"); w_r, f_r = w_r[_o], f_r[_o]
print(f"{SHIFT['rest_files'][0]}: {len(w_r)} px, {w_r[0]:.0f}-{w_r[-1]:.0f} A, "
      f"v_geo = {SHIFT['v_geo_kms']:+.2f} km/s applied")

MARKS = {3933.66: "Ca II K", 3968.47: "Ca II H", 4101.73: r"H$\delta$", 4226.73: "Ca I",
         4340.46: r"H$\gamma$", 4861.35: r"H$\beta$", 5172.70: "Mg b", 5889.95: "Na D",
         6562.80: r"H$\alpha$", 4077.71: "Sr II", 4129.72: "Eu II", 4554.03: "Ba II"}

fig, axes = plt.subplots(2, 1, figsize=(11.5, 6.4))
for ax, (lo, hi) in zip(axes, [(3900, 5300), (5300, 6700)]):
    m = (w_r > lo) & (w_r < hi)
    ax.plot(w_r[m], f_r[m], "k", lw=0.35)
    # stagger the label heights: Sr II, H-delta and Eu II sit within 50 A of each other
    for n_, (lam, lab) in enumerate(sorted((l, b) for l, b in MARKS.items() if lo < l < hi)):
        ax.axvline(lam, color="C3", lw=0.6, ls="--", alpha=0.7, zorder=0)
        ax.text(lam, 1.05 + 0.13 * (n_ % 2), lab, rotation=90, ha="center", va="bottom",
                fontsize=7.5, color="C3")
    ax.set_xlim(lo, hi); ax.set_ylim(0, 1.5)
    ax.set_ylabel("Normalised flux")
    ax.minorticks_on(); ax.tick_params(direction="in", top=True, right=True, which="both")
axes[1].set_xlabel(r"$\lambda$ ($\AA$)")
axes[0].set_title(f"HE 1158-2313 in the rest frame — {SHIFT['v_geo_kms']:+.2f} km/s applied "
                  f"(§4.3, exact Doppler shift)", loc="left", fontsize=10)
fig.tight_layout()
fig.savefig(P + "rest_spectrum_marked.png", dpi=130)
display(fig); plt.close(fig)
print("wrote " + P + "rest_spectrum_marked.png")

### All five spectra together, band by band

In [ ]:
# Same layout as part3's overview_*_vs_star.png - paired rows flush against each other, one
# band per group - but with five rows per band instead of two.
#
# Ours is the REST-frame spectrum here, so all five are in the same frame and the lines should
# coincide; what is left to see is line strength. (The velocity offset itself is the subject of
# the zoom figures above and of part3's two-row overviews.)
BANDS = [(3925, 4045), (4290, 4400), (5140, 5260), (5850, 5960)]
LABEL = {3933.66: "Ca II K", 3968.47: "Ca II H", 4101.73: r"H$\delta$", 4226.73: "Ca I",
         4340.46: r"H$\gamma$", 4861.35: r"H$\beta$", 5172.70: "Mg b", 5889.95: "Na D",
         6562.80: r"H$\alpha$"}

# ordered by how crowded the spectrum is, ours last so it anchors the eye at the bottom
ROWS = [("Sun",                 "Sun",   w_sun,   f_sun,   "0.55"),
        ("HD 122563",           "HD122", w_hd122, f_hd122, "C4"),
        ("HD 140283",           "HD140", w_hd140, f_hd140, "C1"),
        ("G64-12",              "G64",   w_g64,   f_g64,   "C2"),
        ("HE 1158-2313 (rest)", "HE",    w_r,     f_r,     "k")]

fig = plt.figure(figsize=(10.5, 16))
outer = fig.add_gridspec(len(BANDS), 1, hspace=0.20, top=0.955, bottom=0.04,
                         left=0.085, right=0.985)
for b, (lo, hi) in enumerate(BANDS):
    inner = outer[b].subgridspec(len(ROWS), 1, hspace=0.0)
    for k, (name, short, wx, fx, col) in enumerate(ROWS):
        ax = fig.add_subplot(inner[k])
        m = (wx > lo) & (wx < hi)
        xb, yb = wx[m].copy(), fx[m].copy()
        yb[np.r_[np.diff(xb) > 0.5, False]] = np.nan          # never draw across a gap
        ax.plot(xb, yb, color=col, lw=0.5)
        for lam, txt in LABEL.items():
            if lo < lam < hi:
                ax.axvline(lam, color="0.7", lw=0.6, ls="--", zorder=0)
                if k == 0:
                    ax.text(lam, 1.02, txt, ha="center", va="bottom", fontsize=8, color="0.35")
        ax.set_xlim(lo, hi)
        ax.set_ylim(0, 1.32)
        ax.set_yticks([0.0, 0.5, 1.0])
        ax.set_ylabel(short, fontsize=8)
        ax.minorticks_on()
        ax.tick_params(direction="in", top=True, right=True, which="both", labelsize=8)
        if k < len(ROWS) - 1:
            ax.set_xticklabels([])
        elif b == len(BANDS) - 1:
            ax.set_xlabel(r"$\lambda$ ($\AA$)", fontsize=9)
        ax.text(0.995, 0.06, name, transform=ax.transAxes, ha="right", va="bottom",
                fontsize=7.5, color=col,
                bbox=dict(boxstyle="square,pad=0.15", fc="white", ec="none", alpha=0.85))

fig.suptitle("All five spectra at rest — same layout as part3's two-row overviews, five rows per "
             "band.\nLines coincide because every spectrum is in the rest frame; what differs is "
             "depth, i.e. temperature and metallicity.", fontsize=10, x=0.012, ha="left")
fig.savefig(P + "compare_all_systems.png", dpi=130)
display(fig); plt.close(fig)
print("wrote " + P + "compare_all_systems.png")

### The same five rows, now line by line

In [ ]:
# Narrow windows around individual lines, same five-row construction as the band figure.
LINES = [("Ca II K",  3933.66, 6.0),
         ("Mg I",     5183.62, 3.0),
         (r"Na D$_2$", 5889.95, 3.0),
         (r"H$\alpha$", 6562.80, 8.0),
         ("Ba II",    4554.03, 2.5),
         ("Eu II",    4129.72, 2.0)]

fig = plt.figure(figsize=(12.5, 13))
outer = fig.add_gridspec(3, 2, hspace=0.26, wspace=0.16, top=0.905, bottom=0.045,
                         left=0.065, right=0.99)
for g, (lab, lam0, half) in enumerate(LINES):
    inner = outer[g // 2, g % 2].subgridspec(len(ROWS), 1, hspace=0.0)
    lo, hi = lam0 - half, lam0 + half
    for k, (name, short, wx, fx, col) in enumerate(ROWS):
        ax = fig.add_subplot(inner[k])
        m = (wx > lo) & (wx < hi)
        xb, yb = wx[m].copy(), fx[m].copy()
        yb[np.r_[np.diff(xb) > 0.5, False]] = np.nan
        ax.plot(xb, yb, color=col, lw=0.9)
        ax.axvline(lam0, color="0.7", lw=0.7, ls="--", zorder=0)
        # the depth of this line in this star, measured in a +/-0.3 A core window
        cm = (wx > lam0 - 0.3) & (wx < lam0 + 0.3)
        if cm.sum():
            depth = 1 - np.nanmin(fx[cm])
            ax.text(0.015, 0.10, f"{short}  d={depth:.2f}", transform=ax.transAxes,
                    ha="left", va="bottom", fontsize=7.5, color=col,
                    bbox=dict(boxstyle="square,pad=0.15", fc="white", ec="none", alpha=0.85))
        ax.set_xlim(lo, hi)
        ax.set_ylim(0, 1.25)
        ax.set_yticks([0.0, 0.5, 1.0])
        ax.minorticks_on()
        ax.tick_params(direction="in", top=True, right=True, which="both", labelsize=7.5)
        if k < len(ROWS) - 1:
            ax.set_xticklabels([])
        else:
            ax.set_xlabel(r"$\lambda$ ($\AA$)", fontsize=8.5)
        if k == 2:
            ax.set_ylabel("Normalised flux", fontsize=8.5)
        if k == 0:
            ax.set_title(f"{lab} {lam0:.2f} $\\AA$", loc="left", fontsize=9.5)

fig.suptitle("Line by line, all five at rest — rows are Sun, HD 122563, HD 140283, G64-12, "
             "HE 1158-2313.\nd = line depth in a $\\pm$0.3 $\\AA$ core window. Ba II and Eu II are the "
             "r-process lines: strong in ours, essentially absent in the metal-poor comparisons\n"
             "(the Sun's apparent depth at Eu II sits in a crowded region and is mostly "
             "neighbouring blends, not Eu).",
             fontsize=9.5, x=0.012, y=0.985, ha="left", va="top")
fig.savefig(P + "compare_all_systems_lines.png", dpi=130)
display(fig); plt.close(fig)
print("wrote " + P + "compare_all_systems_lines.png")

## Part 4.2.4 (optional) — the G64-12 standard star

In [ ]:
G64 = {t: json.load(open(f"{RES}part4-2-4_g64-12_rv_{t}.json"))
       for t in ("HD140283", "HD122563")}
LATHAM = 443.5          # Latham et al. 2002, heliocentric

print("§4.2.4  G64-12 as the target (the file is already rest-framed):")
for t, d in G64.items():
    print(f"    vs {t}: v = {d['v_geo_kms']:+.2f} km/s")
_g = [d["v_geo_kms"] for d in G64.values()]
print(f"    template-to-template spread : {max(_g) - min(_g):.2f} km/s")
print(f"    largest offset from zero    : {max(abs(v) for v in _g):.2f} km/s")
print(f"    literature (Latham+2002)    : {LATHAM:+.1f} km/s  -> not recovered, as expected")
print()
print(f"    => the method's zero point is good to <= {max(abs(v) for v in _g):.1f} km/s, which is")
print(f"       smaller than the {abs(v_hand[1:].mean() - V_GEO_AF):.1f} km/s by-hand vs cross-correlation difference")

### Results record

In [ ]:
lines = [
 f"HE1158-2313  RA={HDR['RA']} DEC={HDR['DEC']}  {HDR['UT-DATE']}  UT-START={HDR['UT-START']}"
 f"  EXPTIME={HDR['EXPTIME']}s  airmass={HDR['AIRMASS']}",
 "",
 "4.2.1  by hand (Gaussian core fits, this notebook)",
 *[f"    {k:18s} core {v[0]:9.3f} A   v_geo = {v[1]:+8.2f} +/- {v[2]:.2f} km/s"
   f"{'   <- blended with Fe I 5167.49, excluded' if '5167' in k else ''}"
   for k, v in hand.items()],
 f"    clean Mg b mean {v_hand[1:].mean():+.2f} km/s",
 "    redshifted: the star is moving AWAY from us",
 "",
 "4.2.2  cross-correlation",
 *[f"    {t} template, metal-line windows : v_geo = {d['v_geo_kms']:+.2f} +/- {d['v_geo_err_kms']:.2f} km/s"
       f"{'  <- adopted' if t == ADOPTED_TEMPLATE else ''}" for t, d in CCF_ALL.items()],
     f"    H-alpha excluded (broad, chromospheric core, NLTE profile, most template-sensitive);"
     f" it would give {CCF_ALL[ADOPTED_TEMPLATE]['per_window_kms']['H-alpha 6560-6575']:+.2f}",
     f"    template-to-template spread {_spread:.2f} km/s (grid step {CCF['rv_scan_kms'][2]} km/s)",
 f"    grid step {CCF['rv_scan_kms'][2]} km/s, template {CCF['template']}",
 f"    by-hand (clean Mg b) minus cross-correlation: {v_hand[1:].mean() - V_GEO_AF:+.2f} km/s",
 "",
 "4.2.4  G64-12 standard star (optional; the given file is already rest-framed)",
 *[f"    vs {t}: v = {d['v_geo_kms']:+.2f} km/s" for t, d in G64.items()],
 f"    spread {max(_g)-min(_g):.2f} km/s, largest offset from zero {max(abs(v) for v in _g):.2f} km/s",
 f"    literature {LATHAM:+.1f} km/s not recovered -> validates the zero point, not the absolute scale",
 "",
 "4.3    shift to rest",
 f"    applied v_geo = {V_GEO_AF:+.2f} km/s -> {', '.join(SHIFT['rest_files'])}",
 f"    {SHIFT['shift_formula']}",
 "",
 "4.4    heliocentric correction",
 f"    at 00:00 UT (AF's call)  {SHIFT['helio_corr_kms']['at_00UT_AF_call']:+.3f} km/s",
 f"    at UT-START (adopted)    {SHIFT['helio_corr_kms']['at_UT_START']:+.3f} km/s",
 f"    at mid-exposure          {SHIFT['helio_corr_kms']['at_mid_exposure']:+.3f} km/s",
 f"    header velocity keywords: {HDR['_velocity_keywords'] or 'none'}",
 f"    measured {V_GEO_AF:+.2f} vs Gaia DR3 {GAIA:+.2f} +/- {GAIA_E:.2f}  ->  {V_GEO_AF-GAIA:+.2f} km/s  (agrees)",
 f"    measured + correction {V_GEO_AF+HELIO_CORR:+.2f}       ->  {V_GEO_AF+HELIO_CORR-GAIA:+.2f} km/s  (double-counted)",
 "",
 f"ADOPTED  v_helio = {V_GEO_AF:+.2f} +/- {CCF['v_geo_err_kms']:.2f} km/s (metal lines)"
]
open("part4_results.txt", "w").write("\n".join(lines) + "\n")
print("\n".join(lines))